# Queenie — merge correction starter

Hi Queenie! This is your starting point, not a finished model. We load the handover tables, check labels, and measure what happens when we change nothing. Your model choice and training are still yours to do.

**Labels are PROVISIONAL pending group manual review.** Please review a few examples with the group before treating the targets as ground truth. This notebook does not fit a model and never opens test data.

### Keep the boundary clear
- `X` contains only prediction-derived allowlisted features. Reference matches, labels, IDs and split metadata are not model inputs.
- Note features: `start, end, pitch, amplitude, duration, prev_gap, next_gap`; pair features: `gap, pitch_delta, duration_sum`.
- Missing/ambiguous labels are excluded only from this task's training-label diagnostics, never from full end-to-end evaluation.
- Timing targets are absolute seconds; pitch targets are absolute MIDI. Positive classifier labels mean merge the pair / remove the false note.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Open from this notebook folder, or run from the project root.
HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'GTSinger_100_unique_sample'] if (p / 'selected_100.csv').exists())
sys.path.insert(0, str(HERE))
import correction_utils as cu
DATA = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover'
notes = pd.read_csv(DATA / 'train_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
note_targets = pd.read_csv(DATA / 'train_targets.csv', dtype={'note_id':str}).rename(columns={'false_note':'false_label','reference_start':'target_start','reference_end':'target_end','reference_pitch':'target_pitch'})
assert notes['split'].eq('train').all(), 'Only train data belongs in this starter.'
manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
cu.validate_notes(notes, manifest)
features = pd.read_csv(DATA / 'merge_candidates.csv', dtype={'pair_id':str,'recording_id':str,'note_id_a':str,'note_id_b':str}).rename(columns={'note_id_a':'left_note_id','note_id_b':'right_note_id'})
targets = pd.read_csv(DATA / 'merge_targets.csv', dtype={'pair_id':str}).rename(columns={'merge':'merge_label'})
# Pairs must refer to two real notes from the same train recording.
lookup = notes.set_index('note_id')['recording_id']
for side in ['left_note_id','right_note_id']:
    assert features[side].isin(lookup.index).all(), 'Unknown candidate note'
    assert features[side].map(lookup).eq(features['recording_id']).all(), 'Cross-recording candidate'
assert features.left_note_id.ne(features.right_note_id).all()
TASK = 'merge'
X, joined = cu.training_data(features, targets, TASK)
print('Feature allowlist:', X.columns.tolist())
print('Rows:', len(X), '| target-row coverage: 100%')
display(X.head())


Feature allowlist: ['gap', 'pitch_delta', 'duration_sum']
Rows: 0 | target-row coverage: 100%


,gap,pitch_delta,duration_sum


## Check what we can actually learn from

Target rows must cover every feature ID, even when their task labels are missing. For classification, inspect both class counts before choosing a method. For timing/pitch, the value counts below describe target coverage rather than classifier classes.

In [2]:
label_columns = cu.TARGETS[TASK]
usable = joined[label_columns].notna().all(axis=1)
if 'ambiguous' in joined:
    # CSV accepts True/False or 1/0. Reject unknown encodings rather than guessing.
    ambiguity = joined['ambiguous'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    assert ambiguity.notna().all(), 'ambiguous must be explicit True/False or 1/0'
    usable &= ~ambiguity
print({'total_rows':len(joined), 'usable_task_labels':int(usable.sum()),
       'excluded_task_labels':int((~usable).sum())})
for column in label_columns:
    print(column, 'non-null coverage:', float(joined[column].notna().mean()))
    display(joined.loc[usable,column].value_counts(dropna=False).rename('count'))
labelled = joined.loc[usable].copy()
X_task = X.loc[usable].copy()  # Only these allowlisted columns may reach your future model.
folds = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
groups = labelled['sample_id'].copy()
fold_ids = groups.map(folds)
assert fold_ids.notna().all(), 'Missing training-song fold'
print('Labelled examples per song-grouped fold:', fold_ids.value_counts().sort_index().to_dict())
print('Do not fit until the group approves the provisional label version.')


{'total_rows': 0, 'usable_task_labels': 0, 'excluded_task_labels': 0}
merge_label non-null coverage: nan


Series([], Name: count, dtype: int64)

Labelled examples per song-grouped fold: {}
Do not fit until the group approves the provisional label version.


## Our first comparison: do nothing

This is a real no-change baseline on usable training task labels, not evidence of validation performance. Empty-label metrics have no useful interpretation. Nothing is fitted here.

In [3]:
if len(labelled) == 0:
    baseline = {'status':'BLOCKED_NO_REVIEWED_OR_PROVISIONAL_TARGETS', 'n':0, 'metrics':None}
else:
    baseline = cu.classification_metrics(labelled[label_columns[0]], np.zeros(len(labelled), dtype=int))
print('No-change task baseline (not a trained model):', baseline)


No-change task baseline (not a trained model): {'status': 'BLOCKED_NO_REVIEWED_OR_PROVISIONAL_TARGETS', 'n': 0, 'metrics': None}


## Your model goes here — later

Use `X_task` and the separate labels when you are ready. Keep validation separate by song (`sample_id`), choose settings on train/val only, and leave test untouched. Do not infer features from reference alignment. For merge predictions, resolve overlapping pair decisions before producing a complete note table.

## Try the shared output interface

The CSV below contains the real original notes unchanged. A future correction CSV is the **complete replacement note set**, not a patch: omitted notes are deletions; for a merge retain one ID and omit absorbed IDs. Never overwrite the source tables.

In [4]:
# Real, unchanged notes: do not filter this with the task-label mask.
unchanged = notes[cu.NOTE_COLUMNS].copy(deep=True)
output = HERE / 'exports' / 'queenie_merge_unchanged.csv'
if output.exists():
    # Safe notebook rerun: verify the existing export instead of overwriting it.
    existing = pd.read_csv(output, dtype={'note_id':str,'recording_id':str})
    pd.testing.assert_frame_equal(cu.validate_notes(existing, manifest), cu.validate_notes(unchanged, manifest), check_dtype=False)
else:
    cu.export_corrections(unchanged, manifest, output)
print('Unchanged-note interface example:', output)


Unchanged-note interface example: C:\Users\dtpz2\Documents\Daphne documents\projects\datamining\_handover_clean\GTSinger_100_unique_sample\notebooks\correction_handover\exports\queenie_merge_unchanged.csv


## Full end-to-end scoring (when the evaluator tables are ready)

Use `cu.evaluate_end_to_end(corrected, full_references, evaluation_manifest)` for both the unchanged baseline and future corrected notes. All three tables must describe the same declared train/val scope. `evaluation_manifest` has `recording_id, sample_id, split` and must include recordings with zero predictions. `full_references` has `note_id, recording_id, start, end, pitch`; reference IDs are independent of prediction IDs.

Do **not** build this manifest from a filtered prediction table, or filter references with `usable`, task matches, or ambiguity flags. The manifest loaded above is the complete training recording manifest, including zero-prediction recordings. Reference tables are evaluator-only, never feature inputs.

The shared evaluator reuses `bp_tuning_core.score_notes` and `aggregate`: 50 ms onset, 50-cent pitch, optional offsets (20% or 50 ms), song-pooled macro scores and pooled micro F1. The next cell runs it on every training recording using evaluator-only references. It never filters on task-label availability. No validation or test labels are opened.


In [5]:
full_manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv')
end_to_end = cu.evaluate_end_to_end(unchanged, full_references, full_manifest)
print('All-training no-correction baseline:', end_to_end['summary'])
assert end_to_end['summary']['recordings'] == len(full_manifest)


All-training no-correction baseline: {'song_macro_f1': 0.26275846642410683, 'song_macro_offset_f1': 0.15386648702425776, 'song_macro_precision': 0.27330092213557544, 'song_macro_recall': 0.2582440522033118, 'micro_f1': 0.26521100683138604, 'songs': 60, 'recordings': 534}
